# AgenticML — Experimento 1: Iris (Classificação Multiclasse)

**Dataset:** Iris (UCI / sklearn)  
**Tipo de problema:** Classificação multiclasse (3 classes)  
**Objetivo:** Classificar a espécie da flor de íris com base em medidas das pétalas e sépalas  

---
**Contexto:** Este notebook documenta a execução completa do pipeline AgenticML no dataset Iris,
registrando as decisões de cada agente, o modelo selecionado e as métricas obtidas para comparação
com o baseline Auto-sklearn.

In [28]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

# Adiciona raiz do projeto ao path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
_
from core.graph import build_graph
from core.state import initial_state

## 1. Dataset — Iris

| Atributo | Valor |
|----------|-------|
| Instâncias | 150 |
| Features | 4 (sepal_length, sepal_width, petal_length, petal_width) |
| Classes | 3 (setosa, versicolor, virginica) — balanceadas (50 cada) |
| Missing values | Nenhum |
| Fonte | `sklearn.datasets.load_iris()` |

In [29]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.columns = [c.replace(' (cm)', '').replace(' ', '_') for c in df.columns]
df['species'] = iris.target_names[df['target']]
df = df.drop(columns=['target'])

print('Shape:', df.shape)
print('\nDistribuição da target:')
print(df['species'].value_counts())
display(df.head())

Shape: (150, 5)

Distribuição da target:
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [30]:
# Converte para CSV (entrada da API)
csv_bytes = df.to_csv(index=False).encode('utf-8')
print(f'CSV gerado: {len(csv_bytes):,} bytes')

CSV gerado: 3,858 bytes


## 2. Execução do Pipeline AgenticML

In [31]:
PROMPT = (
    "Classificar a espécie da flor de íris (setosa, versicolor ou virginica) "
    "com base nas medidas das pétalas e sépalas."
)

state = initial_state(
    dataset_bytes=csv_bytes,
    filename='iris.csv',
    prompt=PROMPT,
    problem_type_hint='classification',
)

graph = build_graph()

t0 = time.time()
result = graph.invoke(state)
elapsed = time.time() - t0

print(f'\n✓ Pipeline concluído em {elapsed:.2f}s')
print('Status final:', result.get('status'))


✓ Pipeline concluído em 127.38s
Status final: done


In [32]:
# Diagnóstico rápido — execute esta célula antes de inspecionar os resultados por agente
status = result.get('status')
output = result.get('output') or {}
print(f'Status do pipeline: {status}')
if status == 'done' and output.get('status') == 'error':
    # Short-circuit: Agente 6 rodou em modo erro
    print(f"Falhou em: {output.get('failed_at')}")
    print(f"Erro técnico: {(output.get('explanation') or {}).get('technical', '')}")
elif result.get('failed_at'):
    print(f"Falhou em: {result.get('failed_at')}")
    print(f"Erro: {(result.get('error') or {}).get('technical', '')}")
else:
    print('✓ Pipeline concluído com sucesso — todos os agentes executaram.')


Status do pipeline: done
✓ Pipeline concluído com sucesso — todos os agentes executaram.


## 3. Resultados por Agente

### Agente 1 — Ingestão e Inspeção

In [33]:
inspection = result.get('inspection') or {}
schema = inspection.get('schema', {})

print('=== Schema ===')
print('  Colunas:', schema.get('columns'))
print('  Shape:  ', schema.get('shape'))
print('  Tipos:  ', schema.get('dtypes'))
print()
print('Encoding detectado:', inspection.get('encoding_detected'))
print('Separador detectado:', repr(inspection.get('separator_detected')))

print('\n=== Estatísticas Descritivas ===')
stats = inspection.get('descriptive_stats', {})
display(pd.DataFrame(stats).T)

=== Schema ===
  Colunas: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
  Shape:   [150, 5]
  Tipos:   {'sepal_length': 'float64', 'sepal_width': 'float64', 'petal_length': 'float64', 'petal_width': 'float64', 'species': 'str'}

Encoding detectado: utf-8
Separador detectado: ','

=== Estatísticas Descritivas ===


,nulls,mean,std,min,max,median,unique_count,top_values
sepal_length,0.0,5.843333,0.828066,4.3,7.9,5.8,NaN,NaN
sepal_width,0.0,3.057333,0.435866,2.0,4.4,3.0,NaN,NaN
petal_length,0.0,3.758,1.765298,1.0,6.9,4.35,NaN,NaN
petal_width,0.0,1.199333,0.762238,0.1,2.5,1.3,NaN,NaN
species,0,NaN,NaN,NaN,NaN,NaN,3,"{'setosa': 50, 'versicolor': 50, 'virginica': 50}"


### Agente 2 — Diagnóstico de Qualidade

In [34]:
quality = result.get('quality') or {}
issues = quality.get('issues', [])

print(f"Overall severity: {quality.get('overall_severity')}")
print(f"Issues detectados: {len(issues)}")

if issues:
    for i in issues:
        print(f"  [{i['severity'].upper()}] {i['type']}: {i['alert']}")
else:
    print('  ✓ Nenhum problema de qualidade detectado.')

Overall severity: medium
Issues detectados: 2
  [MEDIUM] high_correlation: As colunas 'petal_length' e 'petal_width' apresentam uma alta correlação de 0.9629. Isso pode indicar que uma das colunas é redundante para o modelo de classificação. Recomenda-se avaliar a possibilidade de remover uma dessas colunas para simplificar o modelo sem perder informação significativa.
  [LOW] duplicate_rows: Foi detectada a presença de 1 linha duplicada, representando 0.67% do dataset. Linhas duplicadas podem distorcer o treinamento do modelo. Recomenda-se remover as linhas duplicadas para garantir a integridade dos dados.


### Agente 3 — Feature Engineering

In [35]:
features = result.get('features') or {}
transforms = features.get('transformations_applied', [])

print(f"Coluna target: {features.get('target_column')}")
print(f"Features pós-transformação ({len(features.get('feature_columns', []))}): {features.get('feature_columns')}")
print()
print(f"Transformações aplicadas ({len(transforms)}):")
for t in transforms:
    print(f"  • {t['type']} em {t['columns']}")
    print(f"    → {t['justification']}")

Coluna target: species
Features pós-transformação (4): ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

Transformações aplicadas (1):
  • standard_scaler em ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
    → StandardScaler nas features numéricas para equalizar escalas e evitar dominância em algoritmos sensíveis à magnitude.


### Agente 4 — Interpretação de Linguagem Natural

In [36]:
interp = result.get('interpretation') or {}

print(f"Tipo de problema: {interp.get('problem_type')}")
print(f"Confiança:        {interp.get('confidence')}")
print(f"Divergiu do hint: {interp.get('diverged_from_user')}")
print(f"\nJustificativa:\n  {interp.get('justification')}")

Tipo de problema: classification
Confiança:        high
Divergiu do hint: False

Justificativa:
  O objetivo do usuário é classificar a espécie da flor de íris (setosa, versicolor ou virginica) com base nas medidas das pétalas e sépalas. A coluna target identificada é 'species', que é uma variável categórica com três classes distintas. As colunas de features são medidas numéricas das pétalas e sépalas, que serão usadas para prever a categoria da espécie. Portanto, este é claramente um problema de classificação, pois envolve prever uma categoria discreta com base em variáveis de entrada contínuas.


### Agente 5 — Seleção e Treinamento

In [37]:
training = result.get('training') or {}

print(f"Modelo selecionado: {training.get('model_selected')}")
print(f"Métrica utilizada:  {training.get('metric_used')}")
print(f"Valor da métrica:   {training.get('metric_value')}")
print()
print('Todos os candidatos:')
candidates = training.get('candidate_results', [])
df_cands = pd.DataFrame(candidates).sort_values('score', ascending=False)
display(df_cands)
print(f"\nJustificativa:\n  {training.get('justification')}")

Modelo selecionado: Logistic Regression
Métrica utilizada:  f1_weighted
Valor da métrica:   0.9532

Todos os candidatos:


,model,score
0,Logistic Regression,0.9532
3,SVM,0.9532
1,Random Forest,0.9464
2,XGBoost,0.9389



Justificativa:
  Logistic Regression foi selecionado com F1-score ponderado de 0.9532 para o problema de classification. Demais candidatos avaliados: SVM (0.9532), Random Forest (0.9464), XGBoost (0.9389).


### Agente 6 — Previsão e Explicação

In [38]:
output = result.get('output') or {}
explanation = output.get('explanation', {})
importances = output.get('feature_importances', [])

print('=== Explicação Técnica ===')
print(explanation.get('technical', ''))

print('\n=== Explicação Acessível ===')
print(explanation.get('accessible', ''))

if importances:
    print('\n=== Importância das Features ===')
    display(pd.DataFrame(importances))

=== Explicação Técnica ===
O pipeline AgenticML foi aplicado a um conjunto de dados de 150 linhas e 5 colunas, com o objetivo de classificar a espécie de íris com base em medidas de pétalas e sépalas. O Agente 2 identificou problemas de qualidade, como alta correlação entre variáveis e linhas duplicadas, com severidade geral média. No estágio de feature engineering, o Agente 3 aplicou a transformação standard_scaler nas colunas de medidas para normalizar os dados. O Agente 4 confirmou que o problema é de classificação, pois a variável alvo 'species' é categórica com três classes distintas.

Para o treinamento, o Agente 5 selecionou o modelo de Regressão Logística, que apresentou um F1-score ponderado de 0.9532, igual ao do SVM, mas foi preferido por sua simplicidade e interpretabilidade. Outros modelos avaliados foram Random Forest (0.9464) e XGBoost (0.9389). As features mais importantes identificadas foram 'petal_width' e 'petal_length', com pesos de 1.7595 e 1.5285, respectivamente,

,feature,importance
0,petal_width,1.759517
1,petal_length,1.528503
2,sepal_width,0.773184
3,sepal_length,0.717728


## 4. Resumo

In [39]:
summary = {
    'Experimento':        'Iris — Classificação',
    'Dataset':            f"{schema.get('shape', [0,0])[0]} linhas × {schema.get('shape', [0,0])[1]} colunas",
    'Tipo de problema':   interp.get('problem_type', 'N/A'),
    'Issues de qualidade': len(issues),
    'Transformações':     len(transforms),
    'Modelo selecionado': training.get('model_selected', 'N/A'),
    'Métrica':            training.get('metric_used', 'N/A'),
    'AgenticML score':    training.get('metric_value', 'N/A'),
    'Tempo (s)':          round(elapsed, 2),
}

display(pd.DataFrame([summary]).T.rename(columns={0: 'Valor'}))

,Valor
Experimento,Iris — Classificação
Dataset,150 linhas × 5 colunas
Tipo de problema,classification
Issues de qualidade,2
Transformações,1
Modelo selecionado,Logistic Regression
Métrica,f1_weighted
AgenticML score,0.9532
Tempo (s),127.38
